In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Configurer le WebDriver pour Chrome
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# URL cible
url = "https://vendre.autobiz.fr/liste-garages-reprise/"
driver.get(url)

# Attendre que la section des garages soit chargée
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'list_garage'))
    )
except:
    print("La section 'list_garage' n'a pas été trouvée.")
    driver.quit()
    exit()

# Listes pour stocker les données
garage_names = []
garage_cities = []

# Fonction pour extraire les informations d'une page
def extract_garage_data():
    garages = driver.find_elements(By.CLASS_NAME, 'garage')
    for garage in garages:
        # Extraire le nom du garage
        try:
            name = garage.get_attribute("data-name")  # Récupérer l'attribut data-name
        except:
            name = "N/A"
        
        # Extraire la ville du garage
        try:
            city = garage.find_element(By.CLASS_NAME, 'content-container').find_element(By.TAG_NAME, 'div').text
        except:
            city = "N/A"
        
        # Ajouter les informations aux listes
        garage_names.append(name)
        garage_cities.append(city)

# Initialiser un compteur de pages
page_count = 0
max_pages = 30  # Limiter à 30 pages

# Extraire les données de toutes les pages
while page_count < max_pages:
    # Extraire les données de la page actuelle
    extract_garage_data()

    # Vérifier si un bouton "next" est disponible
    try:
        next_button = driver.find_element(By.ID, 'pagination_next')
        if "disabled" in next_button.get_attribute("class"):
            break  # Quitter si le bouton "next" est désactivé
        next_button.click()  # Passer à la page suivante
        time.sleep(2)  # Attendre le chargement de la page
        page_count += 1  # Incrémenter le compteur de pages
    except:
        break  # Quitter si aucun bouton "next" n'est trouvé

# Fermer le navigateur
driver.quit()

# Vérifier si les données ont été collectées
if garage_names and garage_cities:
    # Créer un DataFrame avec les données extraites
    df = pd.DataFrame({
        'Nom du Garage': garage_names,
        'Ville': garage_cities
    })

    # Exporter les données dans un fichier Excel
    output_path = 'C:/Users/hugoj/Desktop/garages_autobiz.xlsx'
    df.to_excel(output_path, index=False)
    print(f"Données extraites et exportées dans '{output_path}'")
else:
    print("Aucune donnée n'a été extraite.")


Données extraites et exportées dans 'C:/Users/hugoj/Desktop/garages_autobiz.xlsx'
